# Which PyPI packages could `python_exec` install on demand?

The question behind "Packages on demand" (`docs/spec/PYTHON_DIALECT.md`): a model asks for a package, the person
approves, the worker fetches the wheels outside the cell's lockdown and unpacks them. Whether that is worth building
depends on how much of what people use would then work, and whether the approval card can say "this one will not work
here" BEFORE downloading anything.

The runtime is the real one: `pyodide-core/`, the restricted-mode build pinned by `tools/pyodide-dist.json`. Its tags
and marker environment come from that interpreter, not from a table.

1. **Static**: the 2,000 most-downloaded projects, each one's latest release judged by its wheel tags against the
   core's 110 supported tags, with its dependency closure under the core's markers. Projects in the Pyodide lock
   (compiled by Pyodide for this exact ABI) count as installable, as micropip prefers them.
2. **Empirical**: the top 1,000, every one regardless of its static verdict: `micropip.install` in a fresh interpreter,
   then import each module the distribution provides.

Import is the bar measured. A package that imports can still fail when CALLED: sockets (no network inside the cell),
threads, subprocesses, the real filesystem. That is a lower bound on what is broken, and is said so where it matters.

In [1]:
import json, os, re, subprocess, tempfile
from collections import Counter
from pathlib import Path
import pandas as pd
import wmlnb
import sys
sys.path.insert(0, str(wmlnb.ROOT / "notebooks" / "python-packages"))
import pypi

HERE = wmlnb.ROOT / "notebooks" / "python-packages"
DATA = HERE / "data" / "pypi-compat"
N_STATIC, N_PROBE = 2000, int(os.environ.get("PYPKG_N_PROBE", 1000))  # a smaller N_PROBE only while iterating
CACHE = Path(os.environ["PYPKG_CACHE"]) if os.environ.get("PYPKG_CACHE") else None
DIST = json.loads((wmlnb.ROOT / "tools" / "pyodide-dist.json").read_text())
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

def env_probe():
    out = subprocess.run(["node", "notebooks/python-packages/probe.mjs", "env"], cwd=wmlnb.ROOT, capture_output=True, text=True, check=True)
    return [json.loads(out.stdout)]

ENV = wmlnb.extract(DATA / "env.json", env_probe, source={"probe": "probe.mjs env", "core": DIST["artefact"]["name"]})[0]

def static_rows():
    when, top = pypi.top_projects(N_STATIC)
    return [{"top_list_updated": when}] + pypi.classify(top, ENV, CACHE)

_static = wmlnb.extract(DATA / "static.json", static_rows, source={"top": pypi.TOP_URL, "n": N_STATIC, "pypi": "https://pypi.org/pypi/<project>/json"})
TOP_WHEN, static = _static[0]["top_list_updated"], pd.DataFrame(_static[1:])

def probe_rows():
    names = static[static.in_top].sort_values("rank").project.head(N_PROBE).tolist()
    return pypi.probe_installs(names, wmlnb.ROOT)

probes = pd.DataFrame(wmlnb.extract(DATA / "installs.json", probe_rows, source={"probe": "probe.mjs install", "n": N_PROBE, "core": DIST["artefact"]["name"]}))
pd.DataFrame(wmlnb.provenance(DATA / "env.json", DATA / "static.json", DATA / "installs.json"))

,extract,sha256,rows,extracted_at,commit,core,probe,n,pypi,top
0,notebooks/python-packages/data/pypi-compat/env.json,3ccceea77ac38167,1,2026-11-09T08:48:32+00:00,a154fa2828d1,pyodide-restricted-314.0.3-35bd1d69,probe.mjs env,NaN,NaN,NaN
1,notebooks/python-packages/data/pypi-compat/static.json,ad380e3422bc35d6,2225,2026-11-09T09:09:41+00:00,f449ca0a4c5b,NaN,NaN,2000.0,https://pypi.org/pypi/<project>/json,https://hugovk.dev/top-pypi-packages/top-pypi-packages.min.json
2,notebooks/python-packages/data/pypi-compat/installs.json,48d2731c8a139141,1000,2026-11-09T08:45:46+00:00,a154fa2828d1,pyodide-restricted-314.0.3-35bd1d69,probe.mjs install,1000.0,NaN,NaN


In [2]:
pd.Series({"python": ENV["python"], "pyodide": DIST["pyodide"], "abi": DIST["abi"], "core": DIST["artefact"]["name"],
           "supported tags": len(ENV["tags"]), "first tags": ", ".join(ENV["tags"][:3]),
           "lock packages": len(ENV["lock"]), "top list updated": TOP_WHEN,
           "projects judged (top + their deps)": len(static)})

python                                                                                                                   3.14.2
pyodide                                                                                                                 314.0.3
abi                                                                                                                      2026_0
core                                                                                        pyodide-restricted-314.0.3-35bd1d69
supported tags                                                                                                              110
first tags                            cp314-cp314-pyodide_2026_0_wasm32, cp314-cp314-pyemscripten_2026_0_wasm32, cp314-cp314...
lock packages                                                                                                               354
top list updated                                                                                        

## 1. The static verdict over the top 2,000

`kind` is the latest release's own verdict: `lock` (Pyodide builds it for this ABI), `pure-wheel`
(`py3-none-any` and kin), `wasm-wheel` (a compatible emscripten wheel uploaded to PyPI itself, PEP 783),
`no-wasm-build` (only native wheels for other platforms), `sdist-only` (needs a build, never done in the browser),
`python-version` (`requires_python` excludes 3.14). `installable` also needs every dependency in its closure to pass.

In [3]:
top = static[static.in_top].copy()
top["band"] = pd.cut(top["rank"], [0, 100, 500, 1000, 2000], labels=["1-100", "101-500", "501-1000", "1001-2000"])
kinds = pd.crosstab(top["kind"], top["band"], margins=True, margins_name="all")
kinds

band,1-100,101-500,501-1000,1001-2000,all
kind,,,,,
lock,64,78,36,30,208
no-wasm-build,4,35,77,115,231
pure-wheel,32,279,382,824,1517
python-version,0,1,0,6,7
sdist-only,0,1,2,21,24
wasm-wheel,0,6,3,4,13
all,100,400,500,1000,2000


In [4]:
def share(g):
    return pd.Series({"projects": len(g), "installable": g.installable.mean(),
                      "installable, download-weighted": (g.installable * g.downloads_30d).sum() / g.downloads_30d.sum()})
top.groupby("band", observed=True).apply(share, include_groups=False).style.format({"installable": "{:.1%}", "installable, download-weighted": "{:.1%}"})

,projects,installable,"installable, download-weighted"
band,,,
1-100,100.000000,95.0%,97.4%
101-500,400.000000,84.5%,86.8%
501-1000,500.000000,72.4%,73.3%
1001-2000,1000.000000,67.8%,68.2%


### What blocks them

A pure-Python project fails statically when something in its closure is compiled with no build here. The few compiled
projects below block a long tail of pure ones, so each is a candidate for a Pyodide build request, not a reason the
feature cannot exist.

In [5]:
blk = top[(~top.installable) & top.blocked_by.notna()]
blockers = (blk.groupby("blocked_by").agg(blocks=("project", "size"), examples=("project", lambda s: ", ".join(s.head(4))))
               .sort_values("blocks", ascending=False))
blocker_kind = static.set_index("project")["kind"]
blockers["its kind"] = blockers.index.map(lambda b: blocker_kind.get(b, "?"))
blockers.head(15)

,blocks,examples,its kind
blocked_by,,,
grpcio,79,"grpcio-status, opentelemetry-exporter-otlp-proto-grpc, grpc-google-iam-v1, google-clou...",no-wasm-build
psutil,13,"azure-monitor-opentelemetry-exporter, accelerate, pandas-gbq, azure-monitor-opentelemetry",no-wasm-build
pyzmq,13,"jupyter-client, ipykernel, nbclient, nbconvert",no-wasm-build
greenlet,12,"apache-airflow-providers-common-sql, apache-airflow-providers-snowflake, apache-airflo...",no-wasm-build
cryptography,11,"pyopenssl, jwcrypto, asyncssh, webauthn",lock
watchdog,9,"strands-agents, streamlit, mkdocs, mkdocs-material",no-wasm-build
packaging,9,"virtualenv, pre-commit, setuptools-scm, vcs-versioning",lock
cachebox,9,"deepdiff, dbt-adapters, dbt-common, dbt-core",no-wasm-build
ormsgpack,6,"langchain, langgraph, langgraph-checkpoint, langgraph-prebuilt",no-wasm-build


### One wheel per package: the fact a policy matches

A policy names a package by name, version, source and HASH (`docs/spec/POLICIES.md`). That is one pinnable artefact
only when each package in the closure resolves to ONE file under the core's tags. A lock package is one file with its
sha256 in the lock; a PyPI release with several compatible wheels (a pure one and a wasm one, say) leaves micropip a
choice the card would have to show.

In [6]:
by = static.set_index("project")
def closure_of(p):
    seen, stack = {p}, list(by.at[p, "deps"])
    while stack:
        d = stack.pop()
        if d in seen or d not in by.index:
            continue
        seen.add(d); stack += list(by.at[d, "deps"])
    return seen
def one_file(p):
    r = by.loc[p]
    return r.kind == "lock" or r.n_compatible_wheels == 1
inst = top[top.installable]
pinnable = inst.project.map(lambda p: all(one_file(d) for d in closure_of(p)))
multi = static[(static.n_compatible_wheels.fillna(0) > 1) & (static.kind != "lock")]
pd.Series({"installable (top 2000)": len(inst), "every package in the closure is ONE file": int(pinnable.sum()),
           "packages with >1 compatible wheel (any judged)": len(multi),
           "those, by name": ", ".join(multi.project.head(12))})

installable (top 2000)                                                          1473
every package in the closure is ONE file                                        1467
packages with >1 compatible wheel (any judged)                                     4
those, by name                                    isort, multiprocess, backrefs, pex
dtype: object

### Placement: what the host could run that the browser cannot

The spec's "Placement" section runs Python at the host too (CPython there, its numpy and pandas pinned to the lock).
A package with no browser build but a wheel for the host is a PLACEMENT fact, not a "does not work". The host is
assumed to be this laptop: CPython 3.14 on macOS arm64 (`pypi.HOST_TAGS`), not read from a running host.

In [7]:
pd.crosstab(top.installable.map({True: "browser: installable", False: "browser: not"}), top.host.fillna("?"), margins=True)

host,native,none,pure,sdist-only,All
installable,,,,,
browser: installable,118,0,1354,1,1473
browser: not,183,38,270,36,527
All,301,38,1624,37,2000


In [8]:
host_only = top[~top.installable & top.host.isin(["native", "pure"])].sort_values("rank")
", ".join(host_only.project.head(40))

'ruff, greenlet, grpcio, grpcio-status, watchfiles, psutil, uvloop, httptools, virtualenv, hf-xet, psycopg2-binary, tokenizers, pyopenssl, psycopg-binary, langchain, grpcio-tools, pre-commit, uv, asyncpg, rapidfuzz, playwright, opentelemetry-exporter-otlp-proto-grpc, tornado, grpc-google-iam-v1, transformers, snowflake-connector-python, pyzmq, onnxruntime, watchdog, litellm, google-cloud-aiplatform, jupyter-client, setuptools-scm, fastuuid, vcs-versioning, datasets, ruamel-yaml-clib, opentelemetry-exporter-otlp, llvmlite, numba'

In [9]:
nb = top[top.kind.isin(["no-wasm-build", "sdist-only", "python-version"])].sort_values("rank")
nb.groupby("kind").project.apply(lambda s: ", ".join(s.head(25)))

kind
no-wasm-build     ruff, greenlet, grpcio, watchfiles, psutil, uvloop, httptools, hf-xet, psycopg2-binary...
python-version    backports-zstd, gremlinpython, dataclasses, databricks-connect, databricks-labs-dqx, b...
sdist-only        pyspark, docopt, swifter, ratelimit, oss2, pysftp, pycrypto, protobuf3-to-dict, http-e...
Name: project, dtype: str

## 2. The empirical check: install and import the top 1,000

`stage` is where each project stopped: `ok` (installed, every module imported), `import` (installed, some module
raised), `install` (micropip refused or failed), `no-module` (installed, nothing found to import), `timeout`, `crash`.

In [10]:
pr = probes.merge(top[["project", "rank", "downloads_30d", "kind", "installable", "installable_by_tags", "blocked_by", "blocked_why"]],
                  on="project", how="left")
pr["stage"].value_counts().to_frame("projects")

,projects
stage,
ok,725
install,211
import,57
no-module,7


### Does the static verdict predict it?

The card would rest on this: rows are the static `installable`, columns what happened. Off-diagonal cells are where
the filter would be wrong, each listed below with why.

In [11]:
pd.crosstab(pr.installable.map({True: "static: installable", False: "static: not"}), pr.stage, margins=True)

stage,import,install,no-module,ok,All
installable,,,,,
static: installable,56,13,7,719,795
static: not,1,198,0,6,205
All,57,211,7,725,1000


In [12]:
ok = pr.stage == "ok"
tp = (pr.installable & ok).sum(); fp = (pr.installable & ~ok).sum(); fn = (~pr.installable & ok).sum(); tn = (~pr.installable & ~ok).sum()
pd.Series({"precision (static yes -> imports)": tp / (tp + fp), "recall (imports -> static said yes)": tp / (tp + fn),
           "static no, and indeed fails": tn / (tn + fn)}).map("{:.1%}".format)

precision (static yes -> imports)      90.4%
recall (imports -> static said yes)    99.2%
static no, and indeed fails            97.1%
dtype: str

In [13]:
# Static NO, but it installed and imported: micropip found what the tags of the latest release did not show.
surprise = pr[~pr.installable & ok].sort_values("rank")
surprise[["rank", "project", "kind", "blocked_by"]].assign(
    installed=surprise.installed.map(lambda xs: ", ".join(f"{x['name']}=={x['version']} ({x['source']})" for x in xs)[:140])).head(25)

,rank,project,kind,blocked_by,installed
424,425.0,polars-runtime-32,no-wasm-build,NaN,polars-runtime-32==0.0.0 (pypi)
490,491.0,ijson,no-wasm-build,NaN,ijson==3.1.1 (pypi)
600,601.0,claude-agent-sdk,no-wasm-build,NaN,"annotated-types==0.7.0 (pyodide), anyio==4.13.0 (pyodide), attrs==26.1.0 (pyodide), cf..."
604,605.0,grimp,no-wasm-build,NaN,"grimp==2.5 (pypi), typing-extensions==4.15.0 (pyodide)"
732,733.0,dbt-common,pure-wheel,cachebox,"agate==1.9.1 (pypi), attrs==26.1.0 (pyodide), babel==2.18.0 (pypi), certifi==2026.4.22..."
865,866.0,wandb,no-wasm-build,NaN,"annotated-types==0.7.0 (pyodide), certifi==2026.4.22 (pyodide), charset-normalizer==3...."


### Why a project failed, by rule

Every failure of the 1,000 gets ONE cause, decided by the rules below in order, and each cause says whether it
counts against a tag-based filter:

- **version**: micropip refused a version range. A dependency wants a newer version than the lock pins
  (`cryptography>=49` against the lock's 47; `packaging>=26.3` against the 26.1 micropip itself loaded), or two
  dependencies disagree and micropip does not backtrack (`uuid-utils<1.0`). The constraint-aware verdict sees the
  first kind; the second is micropip's.
- **extra**: a dependency asked for an extra (`fastapi[standard]`) whose packages have no build. The static closure
  reads no extras.
- **no build**: a dependency with no wheel for this core.
- **platform**: the code runs into what the core does not have: processes, threads, `resource`, a native shared
  library, a CPU probe, a linux `.so` inside the wheel. A tag cannot show these; only a run can.
- **undeclared host**: the project imports a package it does not declare (an `opentelemetry-instrumentation-*`
  without the library it instruments, a `sphinxcontrib-*` without sphinx). It works once that package is installed:
  not an incompatibility.
- **probe**: the probe's own artefact. It imported a name `top_level.txt` lists that is not a module (`dist`,
  `swagger`), an optional submodule while the main one imported, or found nothing to import (a stub or meta package).

In [14]:
LOCK = {pypi.canonicalize_name(k) for k in ENV["lock"]}
cause = lambda r: pypi.cause(r, LOCK)

COUNTS = {"version": "no, if the filter reads ranges", "extra": "yes (extras unread)", "no build": "yes",
          "platform": "yes, irreducible for tags", "undeclared host": "no", "probe": "no (the probe's)",
          "timeout": "?", "crash": "?", "install: other": "?", "import: other": "?"}
pr["cause"] = pr.apply(lambda r: "ok" if r.stage == "ok" else cause(r), axis=1)
t = pd.crosstab(pr.cause, [pr.installable_by_tags.map({True: "tags: yes", False: "tags: no"}),
                           pr.installable.map({True: "ranges: yes", False: "ranges: no"})])
t["counts against the filter"] = t.index.map(lambda c: COUNTS.get(c, ""))
t

installable_by_tags   tags: no  tags: yes                   counts against the filter
installable         ranges: no ranges: no ranges: yes                                
cause                                                                                
extra                        0          0           3             yes (extras unread)
import: other                1          0           0                               ?
no build                   176          0           1                             yes
ok                           6          0         719                                
platform                     0          0          15       yes, irreducible for tags
probe                        0          0          18                no (the probe's)
undeclared host              0          0          31                              no
version                      7         15           8  no, if the filter reads ranges

**Precision, honestly counted.** A failure the filter could not be blamed for (the probe's own, an undeclared host
package) is removed from both sides; what is left is how often a "yes" on the card would be wrong.

In [15]:
fair = pr[~pr.cause.isin(["probe", "undeclared host"])]
def prec(col):
    yes = fair[fair[col]]
    return pd.Series({"says yes": len(yes), "of those, fail": int((yes.cause != "ok").sum()),
                      "precision": (yes.cause == "ok").mean(),
                      "recall": (fair[fair.cause == "ok"][col]).mean()})
pd.DataFrame({"by tags only": prec("installable_by_tags"), "by tags + version ranges": prec("installable")}).T.style.format(
    {"precision": "{:.1%}", "recall": "{:.1%}"})

,says yes,"of those, fail",precision,recall
by tags only,761.000000,42.000000,94.5%,99.2%
by tags + version ranges,746.000000,27.000000,96.4%,99.2%


In [16]:
fails = pr[pr.installable_by_tags & ~ok].copy()
fails["why"] = fails.cause
fails.why.value_counts().to_frame("statically-installable (by tags) projects that failed")

,statically-installable (by tags) projects that failed
why,
undeclared host,31
version,23
probe,18
platform,15
extra,3
no build,1


In [17]:
def detail(r):
    if r.stage == "install":
        return (r.get("error") or "")[:160]
    bad = [i for i in r.imports if not i["ok"]] if isinstance(r.imports, list) else []
    return f"{bad[0]['module']}: {bad[0]['error_type']}: {bad[0]['error'][:110]} @ {bad[0]['where']}" if bad else r.stage
fails.sort_values("rank").assign(detail=fails.apply(detail, axis=1))[["rank", "project", "why", "detail"]].head(40)

,rank,project,why,detail
104,105.0,soupsieve,undeclared host,soupsieve: ModuleNotFoundError: No module named 'bs4' @ soupsieve/css_match.py:8
133,134.0,virtualenv,version,"Requested 'packaging>=26.3', but packaging==26.1 is already installed. If you want to ..."
165,166.0,pyopenssl,version,"Can't find a pure Python 3 wheel for 'cryptography<51,>=49.0.0'.\nSee: https://pyodide..."
167,168.0,httpx-sse,undeclared host,httpx_sse: ModuleNotFoundError: No module named 'httpx' @ httpx_sse/_api.py:5
169,170.0,psycopg,undeclared host,psycopg: ImportError: no pq wrapper available.\nAttempts made:\n- couldn't import psyc...
191,192.0,google-cloud-bigquery,extra,"Can't find a pure Python 3 wheel for 'grpcio<2.0.0,>=1.75.1; python_version >= ""3.14"" ..."
206,207.0,langchain-core,version,"Requested 'uuid-utils<1.0,>=0.12.0', but uuid-utils==1.2.0 is already installed. If yo..."
210,211.0,pre-commit,version,"Requested 'packaging>=26.3', but packaging==26.1 is already installed. If you want to ..."
219,220.0,gitpython,platform,git: ImportError: Failed to initialize: [Errno 138] emscripten does not support proces...
233,234.0,ptyprocess,platform,ptyprocess: ModuleNotFoundError: No module named 'resource' @ ptyprocess/ptyprocess.py:7


### Import failures across all 1,000, by the module that was missing

A missing stdlib module is a property of the core (Pyodide unvendors some, the restricted fork may drop more); a
missing third-party one is a dependency the metadata did not declare.

In [18]:
miss = Counter()
for r in pr[pr.stage == "import"].itertuples():
    for i in r.imports:
        if not i["ok"]:
            m = re.search(r"No module named '([^']+)'", i["error"])
            miss[(m.group(1).split(".")[0] if m else f"<{i['error_type']}>")] += 1
pd.Series(miss).sort_values(ascending=False).head(20).to_frame("import failures")

,import failures
<OSError>,4
<ImportError>,3
docutils,3
aiohttp,3
sphinx,3
<RuntimeError>,2
dist,2
resource,2
swagger,2
pygments,2


## 3. What an install costs, for the cache

Wheels fetched per successful install, where they came from (the Pyodide CDN for lock packages, PyPI for the rest),
and the time micropip took in Node (network included, so an upper bound for a warm cache).

In [19]:
good = pr[ok].copy()
good["n_installed"] = good.installed.map(len)
good["from_pypi"] = good.installed.map(lambda xs: sum(x["source"] == "pypi" for x in xs))
good["from_lock"] = good.n_installed - good.from_pypi
sizes = static.set_index("project").pypi_wheel_bytes
good["pypi_wheel_mb"] = good.project.map(sizes) / 1e6
good[["n_installed", "from_pypi", "from_lock", "install_s", "pypi_wheel_mb"]].describe(percentiles=[.5, .9, .99]).round(2)

,n_installed,from_pypi,from_lock,install_s,pypi_wheel_mb
count,725.00,725.00,725.00,725.00,725.00
mean,4.29,1.64,2.66,0.50,0.81
std,6.40,2.50,4.55,0.73,2.89
min,0.00,0.00,0.00,0.00,0.00
50%,2.00,1.00,1.00,0.30,0.03
90%,12.00,4.00,9.00,0.87,1.06
99%,25.00,10.00,17.76,3.58,16.43
max,91.00,36.00,55.00,9.02,23.97


In [20]:
src = Counter(x["source"] for xs in good.installed for x in xs)
pd.Series({"wheels installed, total": sum(src.values()), **{f"from {k}": v for k, v in src.items()},
           "installs needing NO PyPI wheel (lock only)": int((good.from_pypi == 0).sum()),
           "installs pulling >20 MB from PyPI": int((good.pypi_wheel_mb > 20).sum())})

wheels installed, total                       3113
from pypi                                     1187
from pyodide                                  1926
installs needing NO PyPI wheel (lock only)     172
installs pulling >20 MB from PyPI                3
dtype: int64

## 4. The version the lock pins vs PyPI's latest

micropip takes the lock's version of a lock package over a newer PyPI release (`requests` installs 2.33.1 from the
lock while PyPI has 2.34.2). A card that shows a version must show the one that will INSTALL.

In [21]:
lk = static[(static.kind == "lock") & static.in_top & static.version.notna()]
lag = lk[lk.version != lk.lock_version]
pd.Series({"lock packages in the top 2000": len(lk), "lock version differs from PyPI latest": len(lag)})

lock packages in the top 2000            208
lock version differs from PyPI latest    161
dtype: int64

In [22]:
lag.sort_values("rank")[["rank", "project", "lock_version", "version"]].rename(columns={"version": "pypi_latest"}).head(15)

,rank,project,lock_version,pypi_latest
1,2.0,packaging,26.1,26.3
2,3.0,typing-extensions,4.15.0,4.16.0
3,4.0,certifi,2026.4.22,2026.10.23
4,5.0,idna,3.11,3.20
5,6.0,urllib3,2.6.3,2.8.0
6,7.0,charset-normalizer,3.4.7,3.5.2
7,8.0,requests,2.33.1,2.34.2
8,9.0,cryptography,47.0.0,50.0.2
9,10.0,cffi,2.0.0,2.1.1
10,11.0,pluggy,1.6.0,1.7.0


## 5. The top 10,000, and the verdicts the card reads

The same probe, run over the top 10,000 (a later ranking than section 2's, frozen for the run) in 20 chunks of 500
queued through test-slot (`pypi.py freeze-top` / `probe-chunk`; a refresh reads the chunks under `PYPKG_10K_DIR`).
Each project gets the cause section 2's rules give it. The result is written to `data/pypi-compat/verdicts.json`,
keyed by the core's release tag, so a later core adds a key instead of overwriting these.

In [23]:
TEN_DIR = Path(os.environ.get("PYPKG_10K_DIR") or Path(tempfile.gettempdir()) / "window-ml-pypi-10k")
_ten = wmlnb.extract(DATA / "probe-10k.json", lambda: pypi.probe_top(10_000, TEN_DIR, wmlnb.ROOT),
                     source={"top": pypi.TOP_URL, "n": 10_000, "probe": "probe.mjs install, 20 chunks of 500", "core": DIST["artefact"]["name"]})
TEN_WHEN, ten = _ten[0]["top_list_updated"], pd.DataFrame(_ten[1:])
pd.DataFrame(wmlnb.provenance(DATA / "probe-10k.json"))

,extract,sha256,rows,extracted_at,commit,core,n,probe,top
0,notebooks/python-packages/data/pypi-compat/probe-10k.json,9c56546389915052,10001,2026-11-09T13:13:21+00:00,539e874c6326,pyodide-restricted-314.0.3-35bd1d69,10000,"probe.mjs install, 20 chunks of 500",https://hugovk.dev/top-pypi-packages/top-pypi-packages.min.json


In [24]:
ten["cause"] = ten.apply(lambda r: "ok" if r.stage == "ok" else cause(r), axis=1)
ten["band"] = pd.cut(ten["rank"], [0, 1000, 2000, 5000, 10_000], labels=["1-1000", "1001-2000", "2001-5000", "5001-10000"])
by_band = pd.crosstab(ten.cause, ten.band, margins=True, margins_name="all")
by_band.loc["loads, share"] = (by_band.loc["ok"] / by_band.loc["all"]).map("{:.0%}".format)
w = ten.groupby("band", observed=True).apply(lambda g: (g.downloads_30d * (g.cause == "ok")).sum() / g.downloads_30d.sum(), include_groups=False)
by_band.loc["loads, download-weighted"] = [f"{x:.0%}" for x in w] + [f"{(ten.downloads_30d * (ten.cause == 'ok')).sum() / ten.downloads_30d.sum():.0%}"]
by_band

band,1-1000,1001-2000,2001-5000,5001-10000,all
cause,,,,,
extra,5,2,9,20,36
import: other,1,9,44,85,139
install: other,0,0,2,6,8
no build,175,265,915,1714,3069
ok,725,612,1682,2350,5369
platform,15,19,29,78,141
probe,18,18,57,56,149
undeclared host,31,35,174,521,761
version,30,40,88,170,328


**Does a second run agree?** The two probes share their top 1,000 (by project, not rank: the rankings are a week
apart and PyPI moved in between). A verdict that changed is a release that changed, or a probe that is not
deterministic.

In [25]:
again = pr[["project", "cause"]].merge(ten[["project", "cause"]], on="project", suffixes=(" (section 2)", " (10k)"))
moved = again[again["cause (section 2)"] != again["cause (10k)"]]
pd.Series({"projects in both": len(again), "same verdict": len(again) - len(moved),
           "moved": "; ".join(f"{r.project}: {r['cause (section 2)']} -> {r['cause (10k)']}" for _, r in moved.iterrows())})

projects in both                                                                                         1000
same verdict                                                                                              996
moved               transformers: version -> no build; google-cloud-aiplatform: no build -> extra; langgra...
dtype: object

**Where the undeclared host packages come from.** The long tail is full of families that extend one package: typing
stubs for boto3's services, `types-*` stubs, instrumentations. They import what they extend without declaring it.

In [26]:
und = ten[ten.cause == "undeclared host"]
def family(p):
    m = re.match(r"(mypy-boto3|types|opentelemetry-instrumentation|sphinxcontrib|django|flake8|pytest|apache-airflow-providers)-", p)
    return m.group(1) + "-*" if m else "(other)"
fam = und.groupby(und.project.map(family)).agg(projects=("project", "size"), first_rank=("rank", "min"), examples=("project", lambda s: ", ".join(s.head(3))))
fam.sort_values("projects", ascending=False)

,projects,first_rank,examples
project,,,
mypy-boto3-*,413,536,"mypy-boto3-s3, mypy-boto3-sqs, mypy-boto3-dynamodb"
(other),191,105,"soupsieve, httpx-sse, psycopg"
types-*,62,1183,"types-aiobotocore-s3, types-boto3-s3, types-boto3-ses"
opentelemetry-instrumentation-*,57,332,"opentelemetry-instrumentation-fastapi, opentelemetry-instrumentation-requests, opentel..."
django-*,23,2897,"django-object-actions, django-pgactivity, django-pglock"
pytest-*,8,648,"pytest-runner, pytest-aiohttp, pytest-freezegun"
sphinxcontrib-*,6,872,"sphinxcontrib-serializinghtml, sphinxcontrib-jsmath, sphinxcontrib-htmlhelp"
flake8-*,1,5259,flake8-plugin-utils


In [27]:
TAG = DIST["artefact"]["tag"]
verdicts_path = DATA / "verdicts.json"
allv = json.loads(verdicts_path.read_text()) if verdicts_path.exists() else {}
allv[TAG] = {"core": DIST["artefact"]["name"], "top_list_updated": TEN_WHEN, "n": len(ten),
             "verdicts": pypi.verdicts(_ten[1:], LOCK)}
body = json.dumps(allv, indent=2, sort_keys=True) + "\n"
if not verdicts_path.exists() or verdicts_path.read_text() != body:
    verdicts_path.write_text(body)
pd.Series(Counter(v["verdict"] for v in allv[TAG]["verdicts"].values())).sort_values(ascending=False).to_frame(f"verdicts.json[{TAG!r}]")

,verdicts.json['pyodide-314.0.3-ml-35bd1d69']
ok,5369
no build,3069
undeclared host,761
version,328
probe,149
platform,141
import: other,139
extra,36
install: other,8


## What this says for "Packages on demand"

The cell below writes this section from the figures above, so the prose cannot drift from them.

In [28]:
from IPython.display import Markdown
band = top.groupby("band", observed=True).apply(lambda g: pd.Series({
    "n": g.installable.mean(), "w": (g.installable * g.downloads_30d).sum() / g.downloads_30d.sum()}), include_groups=False)
pc = lambda x: f"{x:.0%}"
P = prec("installable")
yes = pr[pr.installable]
def asked(r):
    m = re.search(r"Requested '([^']+)', but (\S+) is already installed", r.get("error") or "")
    return f"`{m.group(1)}` against {m.group(2)}" if m else (r.get("error") or "")[:60]
unbacktracked = Counter(asked(r) for _, r in yes[yes.cause == "version"].iterrows())
lock_share = src["pyodide"] / sum(src.values())
not_here = top[~top.installable]
host_ok = not_here.host.isin(["native", "pure"]).sum()
lines = [
    f"Read against `{DIST['artefact']['name']}` (the figures are the cells above):",
    "",
    f"- **Worth building.** Installable by the static verdict: {pc(band.n.iloc[0])} of the top 100 ({pc(band.w.iloc[0])} by "
    f"downloads), {pc(band.n.iloc[1])} of ranks 101-500, {pc(band.n.iloc[3])} of 1001-2000. Of the top {len(pr):,}, "
    f"{int(ok.sum())} installed AND imported in the real core.",
    f"- **The card's 'will this work here' can be decided before downloading**, from PyPI's JSON and the lock: wheel "
    f"tags, the dependency closure under the core's markers, and version ranges checked against the lock (a lock package "
    f"too old for a range blocks only when no release inside the range has a wheel this core accepts; micropip takes the "
    f"newest such release, which is how awscli gets docutils 0.19). Counted fairly (section 2), the verdict is right "
    f"{P['precision']:.1%} of the times it says yes and says yes to {P['recall']:.1%} of what works. What it cannot see: "
    f"platform limits a tag does not carry ({int((yes.cause == 'platform').sum())}: processes, threads, `resource`, native "
    f"shared libraries), micropip not backtracking between two dependencies' ranges "
    f"({int((yes.cause == 'version').sum())}: " + "; ".join(f"{n}x {k}" for k, n in unbacktracked.most_common()) + "), "
    f"and extras ({int((yes.cause == 'extra').sum())}). So the card says 'should work', and the install step still "
    f"reports a failed import as a failure.",
    f"- **Undeclared host packages are a card fact, not a verdict**: {int((pr.cause == 'undeclared host').sum())} plugins "
    f"(`opentelemetry-instrumentation-*`, `sphinxcontrib-*`) import the package they extend without declaring it; the "
    f"card can name it as a second approval.",
    f"- **A policy's hash is one artefact**: {int(pinnable.sum()):,} of the {len(inst):,} installable closures resolve "
    f"to one file per package.",
    f"- **The cache holds lock wheels too**: {lock_share:.0%} of the wheels a successful install fetched came from the "
    f"lock (the Pyodide CDN). The median install fetched {good.n_installed.median():.0f} wheels and "
    f"{good.pypi_wheel_mb.median():.2f} MB from PyPI; the 99th percentile {good.pypi_wheel_mb.quantile(.99):.0f} MB.",
    f"- **The version shown is the one that installs**, usually the lock's: {len(lag)} of the {len(lk)} lock packages in "
    f"the top 2,000 lag PyPI's latest. A package already loaded cannot move at all, so in the extension the start-up "
    f"set pins its versions for every later install.",
    f"- **A few builds unlock a long tail**: `{blockers.index[0]}` alone blocks {int(blockers.blocks.iloc[0])} of the "
    f"top 2,000, then " + ", ".join(f"`{b}`" for b in blockers.index[1:4]) + ".",
    f"- **The host is the other placement**: of the {len(not_here)} projects not installable here, {int(host_ok)} have a "
    f"wheel for a native CPython 3.14 on macOS arm64 (their closure there is not checked).",
    f"- **Over the top {len(ten):,}**, {int((ten.cause == 'ok').sum()):,} install and import "
    f"({(ten.cause == 'ok').mean():.0%}; {(ten.downloads_30d * (ten.cause == 'ok')).sum() / ten.downloads_30d.sum():.0%} "
    f"by downloads), falling from {by_band.loc['loads, share', '1-1000']} of the top 1,000 to "
    f"{by_band.loc['loads, share', '5001-10000']} of ranks 5,001-10,000: no build is "
    f"{int((ten.cause == 'no build').sum()):,} of the failures, and an undeclared host "
    f"{int((ten.cause == 'undeclared host').sum())}, mostly `{fam.projects.idxmax()}`. A second run agrees on "
    f"{len(again) - len(moved)} of the {len(again)} projects both probed. `verdicts.json` holds each one's verdict for "
    f"`{TAG}`, for the card to read.",
    "- **Import is the bar measured**, not use: `requests` installs and imports and still cannot reach the network from "
    "a locked cell. Which calls work is the library manifest's job (`docs/spec/PYTHON_DIALECT.md`).",
]
Markdown("\n".join(lines))

Read against `pyodide-restricted-314.0.3-35bd1d69` (the figures are the cells above):

- **Worth building.** Installable by the static verdict: 95% of the top 100 (97% by downloads), 84% of ranks 101-500, 68% of 1001-2000. Of the top 1,000, 725 installed AND imported in the real core.
- **The card's 'will this work here' can be decided before downloading**, from PyPI's JSON and the lock: wheel tags, the dependency closure under the core's markers, and version ranges checked against the lock (a lock package too old for a range blocks only when no release inside the range has a wheel this core accepts; micropip takes the newest such release, which is how awscli gets docutils 0.19). Counted fairly (section 2), the verdict is right 96.4% of the times it says yes and says yes to 99.2% of what works. What it cannot see: platform limits a tag does not carry (15: processes, threads, `resource`, native shared libraries), micropip not backtracking between two dependencies' ranges (8: 7x `uuid-utils<1.0,>=0.12.0` against uuid-utils==1.2.0; 1x `tenacity<9.2.0,>=8.2.3` against tenacity==9.2.1), and extras (3). So the card says 'should work', and the install step still reports a failed import as a failure.
- **Undeclared host packages are a card fact, not a verdict**: 31 plugins (`opentelemetry-instrumentation-*`, `sphinxcontrib-*`) import the package they extend without declaring it; the card can name it as a second approval.
- **A policy's hash is one artefact**: 1,467 of the 1,473 installable closures resolve to one file per package.
- **The cache holds lock wheels too**: 62% of the wheels a successful install fetched came from the lock (the Pyodide CDN). The median install fetched 2 wheels and 0.03 MB from PyPI; the 99th percentile 16 MB.
- **The version shown is the one that installs**, usually the lock's: 161 of the 208 lock packages in the top 2,000 lag PyPI's latest. A package already loaded cannot move at all, so in the extension the start-up set pins its versions for every later install.
- **A few builds unlock a long tail**: `grpcio` alone blocks 79 of the top 2,000, then `psutil`, `pyzmq`, `greenlet`.
- **The host is the other placement**: of the 527 projects not installable here, 453 have a wheel for a native CPython 3.14 on macOS arm64 (their closure there is not checked).
- **Over the top 10,000**, 5,369 install and import (54%; 85% by downloads), falling from 72% of the top 1,000 to 47% of ranks 5,001-10,000: no build is 3,069 of the failures, and an undeclared host 761, mostly `mypy-boto3-*`. A second run agrees on 996 of the 1000 projects both probed. `verdicts.json` holds each one's verdict for `pyodide-314.0.3-ml-35bd1d69`, for the card to read.
- **Import is the bar measured**, not use: `requests` installs and imports and still cannot reach the network from a locked cell. Which calls work is the library manifest's job (`docs/spec/PYTHON_DIALECT.md`).